В этом ноутбуке собираем eval-набор для RAG по правилам фигурного катания:
- генерируем вопросы из распаршенных документов (`extract_pdf/*.txt`),
- сразу кладём метаданные под гугл-таблицу (Интернациональный/РФ, Вид_катания, Профессиональный, Pdf_name, Сам_запрос, Ожидаемый_ответ и т.п.),
- строим быструю аналитику по покрытию (по документам, дисциплинам, про/любители, типу вопросов).

Пайплайн ниже: загрузка текстов → нарезка на чанки → LLM-генератор → (опционально) LLM-критик → DataFrame → отчёт.

**Как запускать**
1. Установи `OPENAI_API_KEY` (и при желании `OPENAI_MODEL`). Если хочешь обойтись без LLM, поставь `DRY_RUN = True` в конфиге.
2. Прогоняй ячейки сверху вниз. Генерация идёт по стратифицированным чанкам (по 1 из каждого pdf, максимум 80). Настрой `per_pdf`/`max_total` в вызове `stratified_sample`.
3. Для разнообразия по количеству вопросов на запрос используется случайный выбор 1/2/3; фактическое число фиксируется в колонке `Кол-во_вопросов_в_запросе`.
4. Результаты сохраняются в `questions/eval_questions.xlsx` и `.csv` с колонками как в гугл-таблице + служебные поля для аналитики.
5. В аналитике видны покрытия по документам/дисциплинам/странам/аудитории/типам/сложности/answerable и распределение по `Кол-во_вопросов_в_запросе`; flagged-кейсы от критика выводятся отдельно.


In [1]:
import os
import re
import json
import random
import textwrap
from pathlib import Path
from typing import List, Dict, Any


import pandas as pd
from tqdm import tqdm
from openai import OpenAI

PROJECT_ROOT = Path("/home/b.tolstokulakov/study/RAG-AI-Sport-Support")
RAW_TEXT_DIR = PROJECT_ROOT / "extract_pdf"
OUTPUT_DIR = PROJECT_ROOT / "questions"
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = os.environ.get("OPENAI_MODEL", "openai/gpt-4o-mini")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
BASE_URL = os.environ.get("OPENAI_BASE_URL")
MAX_CHARS_PER_CHUNK = 1400        # ограничиваем, чтобы не переполнять prompt
RANDOM_SEED = 13
NUM_QUESTIONS_CHOICES = [1, 2, 3] # хотим разнообразие: 1/2/3 вопроса
DRY_RUN = False                   # True — пропускаем вызовы LLM, генерим заглушки

random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 220)

client = OpenAI(api_key=OPENAI_API_KEY, base_url=BASE_URL) if (OpenAI and OPENAI_API_KEY) else None
client

# Храним уже сгенерированные user_message, чтобы избегать повторов
USER_MESSAGES_SEEN: set[str] = set()


In [2]:
DISCIPLINE_RULES = [
    (r"парн|pair", "парное"),
    (r"танц|dance", "танцы"),
    (r"синхрон|synchro", "синхронное"),
    (r"single|одиноч", "одиночное"),
]

audience_ru = {
    "pro": ["проф", "технические требования", "руководство"],
    "amateur": ["любител", "mass", "amateur"],
    "isu": ["isu", "communication", "handbook", "novice", "component"]
}


def guess_discipline(text: str) -> str:
    low = text.lower()
    for pat, label in DISCIPLINE_RULES:
        if re.search(pat, low):
            return label
    return "общее"


def guess_audience(path: Path, text: str) -> str:
    low = (path.stem + "\n" + text[:800]).lower()
    if any(k in low for k in audience_ru["amateur"]):
        return "любители"
    if any(k in low for k in audience_ru["isu"]):
        return "isu"
    return "профессионалы"


def guess_country(path: Path, text: str) -> str:
    # если много латиницы — считаем международным
    latin_ratio = sum(c.isascii() and c.isalpha() for c in text[:1500]) / max(len(text[:1500]), 1)
    if latin_ratio > 0.5 or "isu" in path.stem.lower():
        return "ISU"
    return "РФ"


def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150) -> List[str]:
    chunks: List[str] = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if c]


def load_raw_docs(raw_dir: Path) -> List[Dict[str, Any]]:
    docs = []
    for path in sorted(raw_dir.glob("*.txt")):
        text = path.read_text(encoding="utf-8", errors="ignore")
        docs.append({
            "pdf_name": path.stem,
            "path": path,
            "text": text,
            "country": guess_country(path, text),
            "audience": guess_audience(path, text),
            "discipline": guess_discipline(path.stem + "\n" + text[:1200]),
        })
    return docs


def build_chunks(docs: List[Dict[str, Any]], chunk_size: int = 900, overlap: int = 150) -> pd.DataFrame:
    rows = []
    for doc in docs:
        for idx, chunk in enumerate(chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)):
            rows.append({
                "chunk_id": f"{doc['pdf_name']}__{idx}",
                "pdf_name": doc["pdf_name"],
                "country": doc["country"],
                "audience": doc["audience"],
                "discipline": doc["discipline"],
                "chunk_text": chunk[:MAX_CHARS_PER_CHUNK],  # обрезаем для промпта
            })
    return pd.DataFrame(rows)


def stratified_sample(df: pd.DataFrame, per_pdf: int = 2, max_total: int = 80) -> pd.DataFrame:
    """Берём по N чанков с pdf, чтобы покрыть все документы."""
    parts = []
    for pdf, group in df.groupby("pdf_name"):
        parts.append(group.sample(n=min(per_pdf, len(group)), random_state=RANDOM_SEED))
    sampled = pd.concat(parts, ignore_index=True)
    if len(sampled) > max_total:
        sampled = sampled.sample(n=max_total, random_state=RANDOM_SEED)
    return sampled.reset_index(drop=True)


docs = load_raw_docs(RAW_TEXT_DIR)
chunks_df = build_chunks(docs, chunk_size=900, overlap=180)
chunks_df.head(3)


,chunk_id,pdf_name,country,audience,discipline,chunk_text
0,2700-ID-Novice-Communication-updated-June2-1759232286-9826__0,2700-ID-Novice-Communication-updated-June2-1759232286-9826,ISU,isu,танцы,"Communication No. 2700\nICE DANCE\nGUIDELINES FOR INTERNATIONAL NOVICE COMPETITIONS 2025/26 (updated June 2, 2025)\nIt is a requirement for certain Technical Rules to be announced annually by the Ice Dance Technical ..."
1,2700-ID-Novice-Communication-updated-June2-1759232286-9826__1,2700-ID-Novice-Communication-updated-June2-1759232286-9826,ISU,isu,танцы,"ree Dance\nFurthermore, the Communication Requirements for Technical Rules with ongoing validity, effective July 1, 2025\nincludes the:\nMarking Guide for GOE for Pattern Dances and Free Dance - Criteria for Levels f..."
2,2700-ID-Novice-Communication-updated-June2-1759232286-9826__2,2700-ID-Novice-Communication-updated-June2-1759232286-9826,ISU,isu,танцы,d the age of\nfourteen (14)\nINTERMEDIATE NOVICE 2 Pattern Dances and Free Dance has not reached the age of sixteen (16)\nADVANCED NOVICE 2 Pattern Dances and Free Dance\nhas reached the age of ten (10) and has not r...


In [3]:
SYSTEM_PROMPT = """Ты генерируешь eval-вопросы для RAG по фигурному катанию.
Требования:
- вопросы должны быть естественными, как от реального пользователя;
- ответ должен вытекать только из переданного контекста (chunk);
- не копируй дословно нумерацию/термины, переформулируй;
- допускаются unanswerable вопросы (если контекст недостаточен), но отметь это флагом.
"""

GEN_PROMPT_TMPL = """Контекст (фрагмент документа):
{context}

Сформируй ОДНО пользовательское сообщение, в котором естественно (как в реальном чате) заданы РОВНО {n_questions} разных вопросов по контексту. Сообщение может содержать 1–2 предложения подводки/мотивации ("я не делал раньше...", "нужно понять, как...") и сами вопросы внутри текста.
Никаких ссылок на документы/страницы/файлы, пользователь не знает источников — пиши как обычный человек.
Выводи РОВНО 1 строку JSONL со структурой:
{{
  "user_message": "...",          # единое пользовательское сообщение с {n_questions} вопросами внутри
  "answers": ["..."],              # список ответов по порядку вопросов, каждый ответ кратко и строго из контекста; если нет ответа — явно пишем, что в контексте нет данных
  "answerable": [true/false],      # список флагов по каждому вопросу
  "question_types": ["fact|procedure|comparison|exception|definition|numeric|multi-hop"],
  "difficulties": ["easy|medium|hard"],
  "supporting_quotes": ["..."]     # по одному короткому фрагменту из контекста на вопрос
}}
Требования:
- user_message должен звучать как живое сообщение, без списков/нумерации и канцелярита.
- Внутри user_message должно быть ровно {n_questions} вопросов (можно через союз/запрос подряд, но без маркдаун-списков).
- Размер списков answers/answerable/question_types/difficulties/supporting_quotes = {n_questions}.
- Выведи только одну строку JSONL без текста вокруг.
"""

CRITIC_PROMPT_TMPL = """Вот вопросы/ответы и исходный контекст.
Проверь для каждой строки: соответствует ли ответ контексту, нет ли галлюцинаций, чётко ли сформулирован вопрос.
Если есть проблема, проставь comment.
Ответь списком JSONL с полями question, verdict (ok|reject|fix), comment.

Контекст:
{context}

Данные:
{data}
"""
def require_client():
    if DRY_RUN:
        return None
    if client is None:
        raise RuntimeError("Нет OPENAI_API_KEY. Установи переменную окружения или включи DRY_RUN=True")
    return client


def llm_chat(messages, **kwargs):
    require_client()
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=kwargs.get("temperature", 0.3),
        max_tokens=kwargs.get("max_tokens", 900),
    )
    return resp.choices[0].message.content


def parse_jsonl(text: str) -> List[Dict[str, Any]]:
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return rows


def generate_for_chunk(row: pd.Series, n_questions: int) -> List[Dict[str, Any]]:
    def ensure_unique_user_message(qa_obj: Dict[str, Any]) -> Dict[str, Any]:
        msg = qa_obj.get("user_message", "").strip()
        if not msg:
            return qa_obj
        if msg in USER_MESSAGES_SEEN:
            # добавим лёгкую вариацию, чтобы избежать дублей
            msg = msg 
            qa_obj["user_message"] = msg
        USER_MESSAGES_SEEN.add(msg)
        return qa_obj

    if DRY_RUN or client is None:
        qa = {
            "user_message": "Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?",  # без упоминаний PDF
            "answers": ["Заглушка. Включи OPENAI_API_KEY для генерации."] * n_questions,
            "answerable": [False] * n_questions,
            "question_types": ["fact"] * n_questions,
            "difficulties": ["easy"] * n_questions,
            "supporting_quotes": [row["chunk_text"][:200]] * n_questions,
        }
        return [ensure_unique_user_message(qa)]

    attempts = 0
    qa_rows: List[Dict[str, Any]] = []
    while attempts < 3:
        prompt = GEN_PROMPT_TMPL.format(
            context=row["chunk_text"],
            n_questions=n_questions,
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        raw = llm_chat(messages)
        qa_rows = parse_jsonl(raw)
        #
        if (
            qa_rows
            and isinstance(qa_rows[0].get("answers", []), list)
            and len(qa_rows[0].get("answers", [])) == n_questions
            and qa_rows[0].get("user_message")
        ):
            qa_rows[0] = ensure_unique_user_message(qa_rows[0])
            break
        attempts += 1

    # если после ретраев всё ещё мало/пусто — дополним шаблоном
    if not qa_rows:
        qa = {
            "user_message": "Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?",  # без упоминаний PDF
            "answers": ["Ответ пока не сгенерирован, дополни вручную."] * n_questions,
            "answerable": [False] * n_questions,
            "question_types": ["fact"] * n_questions,
            "difficulties": ["medium"] * n_questions,
            "supporting_quotes": [row["chunk_text"][:200]] * n_questions,
        }
        qa_rows = [ensure_unique_user_message(qa)]
    else:
        qa = qa_rows[0]
        # добиваем списки до нужной длины, если не хватает
        for key, filler in [
            ("answers", "Ответ пока не сгенерирован, дополни вручную."),
            ("answerable", False),
            ("question_types", "fact"),
            ("difficulties", "medium"),
            ("supporting_quotes", row["chunk_text"][:200]),
        ]:
            arr = qa.get(key, []) or []
            while len(arr) < n_questions:
                arr.append(filler)
            qa[key] = arr[:n_questions]
        qa_rows = [ensure_unique_user_message(qa)]

    return qa_rows

def critic_pass(row: pd.Series, qa_rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if DRY_RUN or client is None:
        return qa_rows

    # добавляем идентификатор вопроса для сопоставления
    for qa in qa_rows:
        if "question" not in qa:
            qa["question"] = qa.get("user_message", "")

    data = "\n".join(json.dumps(x, ensure_ascii=False) for x in qa_rows)
    prompt = CRITIC_PROMPT_TMPL.format(context=row["chunk_text"], data=data)
    messages = [
        {"role": "system", "content": "Ты проверяешь соответствие ответов контексту."},
        {"role": "user", "content": prompt},
    ]
    raw = llm_chat(messages, temperature=0.0)
    reviews = parse_jsonl(raw)

    comments = {r.get("question"): r for r in reviews}
    updated = []
    for qa in qa_rows:
        review = comments.get(qa.get("question"), {})
        if review.get("verdict") in {"reject", "fix"}:
            qa["logic_issue"] = review.get("comment", "flagged by critic")
        # эвристика: если supporting_quote не находится в контексте, помечаем
        if not qa.get("logic_issue"):
            quotes = qa.get("supporting_quotes") or qa.get("supporting_quote") or []
            ctx = row["chunk_text"]
            if any(q and q not in ctx for q in quotes):
                qa["logic_issue"] = "supporting_quote not found in chunk"
        # эвристика по числам: если ответ содержит числа, которых нет в контексте, флаг
        if not qa.get("logic_issue"):
            answers = qa.get("answers") or []
            nums = re.findall(r"\d+", " ".join(str(a) for a in answers))
            if nums and any(n not in row["chunk_text"] for n in nums):
                qa["logic_issue"] = "numeric answer not in chunk"
        updated.append(qa)
    return updated

In [4]:
sample_chunks = stratified_sample(chunks_df, per_pdf=3, max_total=70)
print(f"Всего чанков: {len(chunks_df)}, будем генерить по {len(sample_chunks)} выбраным." )

records = []
for _, row in tqdm(sample_chunks.iterrows(), total=len(sample_chunks), desc="Генерация по чанкам"):
    n_questions = random.choice(NUM_QUESTIONS_CHOICES)
    qa_rows = generate_for_chunk(row, n_questions=n_questions)
    qa_rows = critic_pass(row, qa_rows)
    qa = qa_rows[0]
    actual_n = len(qa.get("answers", []))
    records.append({
        "Интернациональный/РФ": row["country"],
        "Вид_катания": row["discipline"],
        "Профессиональный": row["audience"],
        "Pdf_name": row["pdf_name"],
        "Ошибка_в_логике": qa.get("logic_issue", ""),
        "Кол-во_вопросов_в_запросе": actual_n,
        "Сам_запрос": qa.get("user_message"),
        "Ожидаемый_ответ": "\n".join(qa.get("answers", [])),
        # доп. признаки для аналитики/отладки
        "answerable": qa.get("answerable"),
        "question_type": qa.get("question_types"),
        "difficulty": qa.get("difficulties"),
        "supporting_quote": qa.get("supporting_quotes"),
        "chunk_id": row["chunk_id"],
    })

eval_df = pd.DataFrame(records)
print(eval_df.shape)
eval_df.head(5)


Всего чанков: 4320, будем генерить по 70 выбраным.


Генерация по чанкам:   0%|          | 0/70 [00:00<?, ?it/s]

Генерация по чанкам: 100%|██████████| 70/70 [13:30<00:00, 11.57s/it]

(70, 13)


,Интернациональный/РФ,Вид_катания,Профессиональный,Pdf_name,Ошибка_в_логике,Кол-во_вопросов_в_запросе,Сам_запрос,Ожидаемый_ответ,answerable,question_type,difficulty,supporting_quote,chunk_id
0,РФ,танцы,любители,ПРЕДВАРИТЕЛЬНЫЕ_Техтребования_Танцы_соло_2025_2026,"Вопрос не содержит конкретной информации о том, какие именно моменты интересуют пользователя, что делает его слишком общим.",2,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Ответ пока не сгенерирован, дополни вручную.\nОтвет пока не сгенерирован, дополни вручную.","[False, False]","[fact, fact]","[medium, medium]","[ребованиям, получат звездочку (*) и, следовательно, БЕЗ | К |\n| | может быть без , ребованиям, ...",ПРЕДВАРИТЕЛЬНЫЕ_Техтребования_Танцы_соло_2025_2026__42
1,РФ,парное,профессионалы,Руководство Технических бригад Парное катание,"Вопрос не конкретен и не соответствует контексту, так как не указывает, какие именно моменты интересуют пользователя.",2,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Ответ пока не сгенерирован, дополни вручную.\nОтвет пока не сгенерирован, дополни вручную.","[False, False]","[fact, fact]","[medium, medium]","[ию, то будет названа комбинация прыжков.\n\n\n--- Table 34 ---\n| Падение/касание | Как называть , ию, то бу...",Руководство Технических бригад Парное катание__150
2,ISU,танцы,isu,PATTERNDANCES202324NOVICE070475800_17327128449300,,3,"Я только начинаю разбираться в фигурном катании и у меня есть несколько вопросов. Как называется последняя мелодия, которую используют для разминки? Сколько всего мелодий в этом наборе? И есть ли информация о том, кт...",последняя мелодия для разминки не указана\nв наборе 6 мелодий\nв контексте нет данных,"[True, True, False]","[fact, numeric, unanswerable]","[easy, easy, medium]","[the 6th (last) tune of the, the 6th (last) tune of the, ]",PATTERNDANCES202324NOVICE070475800_17327128449300__33
3,РФ,общее,профессионалы,Хореографическая спираль,"Ответ не сгенерирован, и вопрос не содержит конкретной информации для ответа.",3,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Ответ пока не сгенерирован, дополни вручную.\nОтвет пока не сгенерирован, дополни вручную.\nОтвет пока не сгенерирован, дополни вручную.","[False, False, False]","[fact, fact, fact]","[medium, medium, medium]","[хорошей энергией и плавностью исполнения 3. Наличие глубоких рёбер, контроль всего тела 4. Хорошие ясность и точность движений 5. Необычность и/или оригинальность\nДля + 4 и + 5 ПЕРВЫЕ ТРИ ПУНКТА, выде, хорошей энер...",Хореографическая спираль__3
4,ISU,парное,isu,TP-Handbook-Pair-Skating-2025-26-25July-1754657063-3208,"Вопрос нечеткий и не содержит конкретной информации о том, какие именно моменты интересуют пользователя. Ответ не сгенерирован, что также не соответствует ожиданиям.",1,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Ответ пока не сгенерирован, дополни вручную.",[False],[fact],[medium],"[n\nIf any spin does not have at least 2 continuous revolutions in a basic position, no Level has to be given.\nLess than 2 revolutions in basic positions\nA spin combination executed with only 1 position]",TP-Handbook-Pair-Skating-2025-26-25July-1754657063-3208__51


In [5]:
def coverage(df: pd.DataFrame, by: str) -> pd.DataFrame:
    return (
        df.groupby(by)
        .size()
        .reset_index(name="cnt")
        .sort_values("cnt", ascending=False)
        .reset_index(drop=True)
    )

summary = {
    "docs": coverage(eval_df, "Pdf_name"),
    "discipline": coverage(eval_df, "Вид_катания"),
    "audience": coverage(eval_df, "Профессиональный"),
    "country": coverage(eval_df, "Интернациональный/РФ"),
    "question_type": coverage(eval_df.explode("question_type"), "question_type"),
    "difficulty": coverage(eval_df.explode("difficulty"), "difficulty"),
    "answerable": coverage(eval_df.explode("answerable"), "answerable"),
    "num_questions": coverage(eval_df, "Кол-во_вопросов_в_запросе"),
}

for name, table in summary.items():
    print(f"\n=== {name} ===")
    display(table.head(20))

# быстрые QA-кейсы с пометками критика
flagged = eval_df[eval_df["Ошибка_в_логике"] != ""]
print(f"\nПроблемных по критику: {len(flagged)}")
if not flagged.empty:
    display(flagged[["Pdf_name", "Сам_запрос", "Ошибка_в_логике"]].head(10))

# сохраним под гугл-таблицу и резерв с доп. полями
excel_path = OUTPUT_DIR / "eval_questions.xlsx"
csv_path = OUTPUT_DIR / "eval_questions.csv"
eval_df.to_excel(excel_path, index=False)
eval_df.to_csv(csv_path, index=False)
print(f"Сохранено в {excel_path} и {csv_path}")



=== docs ===


,Pdf_name,cnt
0,2700-ID-Novice-Communication-updated-June2-1759232286-9826,3
1,2716---ID-Levels---Requirements-for-Technical-Rules-2025-26-updated-Aug13-1759232286-5633,3
2,Componentschartupdated2024July086344800_17327068456450,3
3,EXPLANATIONOFSYMBOLSONTHEJUDGESDETAILSPERSKATERNovice030044900_17327128473531,3
4,Figure_Skating_Media_Guide_2024-25,3
5,KeyPointsandKeyPointsFeaturesforJuniorPatternDanceElementsSeason202324AdjustmentsfromCommunication2560RockerFoxtrot043040100_17327128465944,3
6,PATTERNDANCES202324NOVICE070475800_17327128449300,3
7,SPComponentCharts2024064619000_17327068477733,3
8,SPComponentCharts20240387849001741256145-036,3
9,TP-Handbook-Singles-25-26-FINAL-21-July-2025-update-25-July-1753703999-2708,3



=== discipline ===


,Вид_катания,cnt
0,парное,25
1,танцы,21
2,общее,14
3,одиночное,7
4,синхронное,3



=== audience ===


...


=== num_questions ===


,Кол-во_вопросов_в_запросе,cnt
0,3,26
1,2,24
2,1,20



Проблемных по критику: 59


,Pdf_name,Сам_запрос,Ошибка_в_логике
0,ПРЕДВАРИТЕЛЬНЫЕ_Техтребования_Танцы_соло_2025_2026,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Вопрос не содержит конкретной информации о том, какие именно моменты интересуют пользователя, что делает его слишком общим."
1,Руководство Технических бригад Парное катание,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Вопрос не конкретен и не соответствует контексту, так как не указывает, какие именно моменты интересуют пользователя."
3,Хореографическая спираль,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Ответ не сгенерирован, и вопрос не содержит конкретной информации для ответа."
4,TP-Handbook-Pair-Skating-2025-26-25July-1754657063-3208,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Вопрос нечеткий и не содержит конкретной информации о том, какие именно моменты интересуют пользователя. Ответ не сгенерирован, что также не соответствует ожиданиям."
5,Хореографическая спираль,"Я только начинаю разбираться в фигурном катании и мне нужно понять, что такое хореографическая спираль? Как долго должна скользить нога, чтобы позиция была засчитана? И чем отличаются спирали на одноименных ногах от ...",supporting_quote not found in chunk
6,ПРАВИЛА ВИДА СПОРТА фигурное катание,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Вопрос не конкретен и не содержит информации о том, какие именно моменты интересуют пользователя."
8,Руководство технических бригад Синхронное катание,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Вопрос неясен и не конкретен. Не указано, какие именно моменты интересуют пользователя."
9,ПРЕДВАРИТЕЛЬНЫЕ_Технические_требования_2025_2026_послед,"Я только начинаю разбираться в фигурном катании и хотел бы понять, какие навыки оцениваются в хореографической последовательности?",supporting_quote not found in chunk
10,Figure_Skating_Media_Guide_2024-25,"Я недавно начал интересоваться фигурным катанием и хотел бы узнать, кто из фигуристов представлял Японию, и какие города упоминаются в контексте. Также интересно, сколько раз упоминается имя Эвана Лисека. Может, ты з...",supporting_quote not found in chunk
12,ПРЕДВАРИТЕЛЬНЫЕ_Технические_требования_для_детей_любителей_2025_2026,Привет! Нужна помощь: можешь подсказать по этим правилам пару моментов?,"Ответ не сгенерирован, что делает его неполным и неинформативным."


Сохранено в /home/b.tolstokulakov/study/RAG-AI-Sport-Support/questions/eval_questions.xlsx и /home/b.tolstokulakov/study/RAG-AI-Sport-Support/questions/eval_questions.csv


**Важно про формат запросов**
- Теперь `Сам_запрос` — это одно пользовательское сообщение, внутри которого ровно `Кол-во_вопросов_в_запросе` вопросов.
- В `Ожидаемый_ответ` кладём ответы на все вопросы (по порядку) через перевод строки.
- Служебные поля (`answerable`, `question_type`, `difficulty`, `supporting_quote`) теперь списки той же длины, что и число вопросов в сообщении.


In [6]:
eval_df.shape

(70, 13)

In [7]:
eval_df['Ошибка_в_логике']

0                                               Вопрос не содержит конкретной информации о том, какие именно моменты интересуют пользователя, что делает его слишком общим.
1                                                     Вопрос не конкретен и не соответствует контексту, так как не указывает, какие именно моменты интересуют пользователя.
2                                                                                                                                                                          
3                                                                                             Ответ не сгенерирован, и вопрос не содержит конкретной информации для ответа.
4     Вопрос нечеткий и не содержит конкретной информации о том, какие именно моменты интересуют пользователя. Ответ не сгенерирован, что также не соответствует ожиданиям.
                                                                                      ...                                                   

In [1]:
1+1

2